# OpenAI framework example (compare offers)

Follow up the readme.md to get started

In [24]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from openai.types.responses import ResponseTextDeltaEvent
from openai import OpenAI
from pydantic import BaseModel, Field
import os
import asyncio


load_dotenv(override=True)

True

In [25]:
offer_names = ["prosermo", "enpal", "arndt", "widmann"]

In [26]:
# Generator function. Yields result to calling function
def handle_files_generator(files):
    for filename in files:
        # This is not printing the result
        yield filename + ".txt"

def read_offer_file(filename):
    with open(filename, 'r') as file:
        return file.read()

# Call generator as an iterator and print the result
for file in handle_files_generator(offer_names):
    print(file)

prosermo.txt
enpal.txt
arndt.txt
widmann.txt


In [27]:
class EnergyOffer(BaseModel):
    concerns_PV: bool = Field(description="Whether the offer is related to photovoltaics (PV)")
    concerns_WP: bool = Field(description="Whether the offer is related to heat pumps (WP)")
    PV_offer: str = Field(description="The photovoltaic (PV) related offer details")
    WP_offer: str = Field(description="The heat pump (WP) related offer details")

EnergyOffer.model_json_schema()

{'properties': {'concerns_PV': {'description': 'Whether the offer is related to photovoltaics (PV)',
   'title': 'Concerns Pv',
   'type': 'boolean'},
  'concerns_WP': {'description': 'Whether the offer is related to heat pumps (WP)',
   'title': 'Concerns Wp',
   'type': 'boolean'},
  'PV_offer': {'description': 'The photovoltaic (PV) related offer details',
   'title': 'Pv Offer',
   'type': 'string'},
  'WP_offer': {'description': 'The heat pump (WP) related offer details',
   'title': 'Wp Offer',
   'type': 'string'}},
 'required': ['concerns_PV', 'concerns_WP', 'PV_offer', 'WP_offer'],
 'title': 'EnergyOffer',
 'type': 'object'}

In [ ]:
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5.4-mini")
BASE_URL = os.getenv("BASE_URL", "https://api.openai.com/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

AI_client = OpenAI(base_url=BASE_URL, api_key=OPENAI_API_KEY)
settings = ModelSettings(model=MODEL_NAME, base_url=BASE_URL, api_key=OPENAI_API_KEY)
instructions = "You route the offers depending on their content (PV or WP)."

content_router = Agent(name="content_router", instructions=instructions, output_type=EnergyOffer, model_settings=settings)

with trace("Routing offers based on content"):
    for offer_name in handle_files_generator(offer_names):
        result = await Runner.run(content_router, read_offer_file(".docs/" + offer_name))
        print(f"Offer: {offer_name}")
        print(f"Routing Result: PV is {result.final_output.concerns_PV}, WP is {result.final_output.concerns_WP}")
        print(f"PV Offer Details: {result.final_output.PV_offer}")
        print(f"WP Offer Details: {result.final_output.WP_offer}")


FileNotFoundError: [Errno 2] No such file or directory: 'docs/prosermo.txt'